# HandScribe XR: generate EPIC-KITCHENS outputs on a free Colab GPU

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. Set `PART` in the next cell. Each person runs a different part, e.g. `"1/2"` and `"2/2"`.
3. **Runtime → Run all.** Approve the Google Drive prompt. Keep this tab open.

Every finished video is saved immediately as a JSON file in your Google Drive, in the folder `track04_outputs`.
If Colab disconnects or the 12-hour limit is reached, just **Run all** again. Finished videos are skipped
and the run continues where it stopped.

In [ ]:
PART = "1/2"            # "1/2" for one person, "2/2" for the other (or 1/3, 2/3, 3/3 for three)
TEAM_ID = "404NotFound"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU: Runtime -> Change runtime type -> T4 GPU"

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/track04_outputs"
os.makedirs(OUT, exist_ok=True)
print(len([f for f in os.listdir(OUT) if f.endswith(".json")]), "outputs already in Drive")

In [ ]:
%cd /content
![ -d Samsung ] || git clone -q --depth 1 https://github.com/khushichhabra7921/Samsung.git
%cd /content/Samsung
!git pull -q
!apt-get -qq install -y libegl1 libgles2 libportaudio2 > /dev/null
!pip -q install uv
![ -x .venv/bin/python ] || uv venv -q --python 3.12 .venv
!uv pip install -q --python .venv/bin/python torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu126
!uv pip install -q --python .venv/bin/python -r requirements.txt
!.venv/bin/python tools/download_weights.py
!.venv/bin/python -c "import torch; print('CUDA ready:', torch.cuda.is_available())"

The next cell runs for hours and prints one line per video (`done in … s`).
At the end it reports `problem group(s)` for the *other* parts' videos, which is expected.

In [ ]:
!HANDSCRIBE_DEVICE=cuda HANDSCRIBE_TEAM_ID={TEAM_ID} .venv/bin/python -u tools/generate_outputs.py --dataset epic --part {PART} --fetch /content/videos --outputs {OUT} 2>&1 | grep --line-buffered -vE "W0000|I0000|INFO:|Loading weights|transformers\]|NORM_RECT|Feedback manager"

**When it says `0 failed`:** open Google Drive, right-click `track04_outputs` → **Download**, and send the zip to Chintan.